In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
from datetime import datetime

st.title("Otomatik Gayrimenkul Değerleme Paneli")
st.caption("CatBoost makine öğrenmesi modeli ile taşınmazın tahmini satış değeri ve yatırım geri dönüş süresi hesaplanır.")

if 'df' not in st.session_state or 'models' not in st.session_state:
    st.warning("Lütfen önce ana sayfayı (app.py) çalıştırınız.")
    st.stop()

df = st.session_state['df']
models = st.session_state['models']
model_mid, model_low, model_high = models

st.markdown("""
<div class="content-card">
    <h4>Taşınmaz Parametre Girişi (İstanbul)</h4>
    Değerlemesi yapılacak taşınmazın fiziksel ve konum bilgilerini giriniz.
</div>
""", unsafe_allow_html=True)

c_deg1, c_deg2, c_deg3 = st.columns(3)
with c_deg1:
    st.selectbox("İl", ["İstanbul"], key="val_il_sabit")
    s_ilce_options = sorted(df['İlçe'].unique().tolist())
    s_ilce = st.selectbox("İlçe", s_ilce_options, key="val_ilce")
with c_deg2:
    s_brut = st.number_input("Brüt Kullanım Alanı (m²)", 30, 800, 120, key="val_brut")
    s_oda = st.selectbox("Oda Sayısı", [1, 2, 3, 4, 5, 6, 7, 8], index=2, key="val_oda")
    s_salon = st.selectbox("Salon Sayısı", [0, 1, 2], index=1, key="val_salon")
with c_deg3:
    s_bina_yasi = st.number_input("Bina Yaşı (Yıl)", 0, 60, 5, key="val_yas")

st.markdown("##### Ek Yapı Nitelikleri")
sc1, sc2, sc3 = st.columns(3)
with sc1:
    s_deprem = st.checkbox("Deprem Yönetmeliğine Uygun", key="v_deprem")
    s_site = st.checkbox("Site İçerisinde", key="v_site")
with sc2:
    s_havuz = st.checkbox("Yüzme Havuzu Mevcut", key="v_havuz")
    s_otopark = st.checkbox("Otopark Mevcut", key="v_otopark")
with sc3:
    s_ebeveyn = st.checkbox("Ebeveyn Banyosu Mevcut", key="v_ebeveyn")
    s_sifir = st.checkbox("Sıfır / Yeni Yapı", key="v_sifir")

st.markdown("<br>", unsafe_allow_html=True)
if st.button("Değerleme Raporunu Hesapla", type="primary", use_container_width=True):
    benzer = df[df['İlçe'] == s_ilce]['Semt'].mode()
    semt_degeri = benzer.iloc[0] if len(benzer) > 0 else s_ilce

    toplam_o = s_oda + s_salon
    m2_per_o = s_brut / max(toplam_o, 1)

    input_data = pd.DataFrame([{
        'm² (Brüt)': s_brut,
        'oda_sayisi': float(s_oda),
        'salon_sayisi': float(s_salon),
        'm2_per_oda': m2_per_o,
        'bina_yasi': float(s_bina_yasi),
        'Semt': semt_degeri
    }])

    pred_mid = float(np.expm1(model_mid.predict(input_data)[0]))
    pred_low = float(np.expm1(model_low.predict(input_data)[0]))
    pred_high = float(np.expm1(model_high.predict(input_data)[0]))
    
    multiplier = 1.0
    if s_deprem: multiplier *= 1.05
    if s_site: multiplier *= 1.04
    if s_havuz: multiplier *= 1.03
    if s_otopark: multiplier *= 1.02
    if s_ebeveyn: multiplier *= 1.02
    if s_sifir: multiplier *= 1.05

    pred_mid *= multiplier
    pred_low *= multiplier
    pred_high *= multiplier

    st.success("Değerleme Hesaplaması Tamamlandı")
    
    m1, m2, m3 = st.columns(3)
    m1.metric("Tahmini Piyasa Bedeli (%50)", f"{pred_mid:,.0f} TL")
    m2.metric("Hızlı Satış Alt Bandı (%10)", f"{pred_low:,.0f} TL")
    m3.metric("Maksimum Satış Bandı (%90)", f"{pred_high:,.0f} TL")

    tahmini_aylik_kira = pred_mid / 220
    amortisman_yil = pred_mid / (tahmini_aylik_kira * 12)
    yillik_getiri_yuzde = ((tahmini_aylik_kira * 12) / pred_mid) * 100

    st.divider()
    st.markdown("### Yatırım ve Amortisman Analizi")
    y1, y2, y3 = st.columns(3)
    y1.metric("Tahmini Aylık Kira Potansiyeli", f"{tahmini_aylik_kira:,.0f} TL/Ay")
    y2.metric("Amortisman Süresi", f"{amortisman_yil:.1f} Yıl")
    y3.metric("Yıllık Brüt Getiri Oranı", f"%{yillik_getiri_yuzde:.2f}")

    rapor_metni = f"""GAYRİMENKUL DEĞERLEME VE YATIRIM RAPORU
Tarih: {datetime.now().strftime('%d/%m/%Y %H:%M')}

1. TAŞINMAZ BİLGİLERİ
-----------------------------------------
Lokasyon: İstanbul / {s_ilce}
Brüt Alan: {s_brut} m²
Bina Yaşı: {s_bina_yasi} Yıl
Oda Yapısı: {s_oda}+{s_salon}

2. DEĞERLEME SONUÇLARI (CATBOOST REGRESSION)
-----------------------------------------
Tahmini Piyasa Satış Bedeli: {pred_mid:,.0f} TL
Alt Bant (%10 Quantile / Hızlı Satış): {pred_low:,.0f} TL
Üst Bant (%90 Quantile / Tavan Satış): {pred_high:,.0f} TL

3. YATIRIM VE AMORTİSMAN ANALİZİ
-----------------------------------------
Tahmini Aylık Kira Potansiyeli: {tahmini_aylik_kira:,.0f} TL / Ay
Amortisman Süresi: {amortisman_yil:.1f} Yıl
Yıllık Brüt Getiri Oranı: %{yillik_getiri_yuzde:.2f}
"""
    st.download_button("Değerleme Raporunu Metin Dosyası Olarak İndir", data=rapor_metni, file_name=f"Gayrimenkul_Degerleme_Raporu_{s_ilce}_{datetime.now().strftime('%Y%m%d')}.txt", mime="text/plain")